# Phase 01 - Data understanding

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded!")

Libraries loaded!


### Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Load the Dataset

In [3]:
# Update this path to match your actual folder/file name in Google Drive
file_path = "/content/drive/MyDrive/Retail Sales & Customer Insights Project/Dataset/online_retail_II.csv"

# encoding="ISO-8859-1" helps read special characters like the £ symbol correctly
df = pd.read_csv(file_path, encoding="ISO-8859-1")

print("Data loaded successfully!")
print("Shape (rows, columns):", df.shape)

Data loaded successfully!
Shape (rows, columns): (1067371, 8)


### Discribe the Dataset

First Five Row

In [4]:
# Shows the first 5 rows of the dataset
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Check Data Types

In [5]:
# Shows column names, data types, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


Statistical Summary

In [6]:
# Shows count, mean, min, max, etc. for numeric columns
# Check if Quantity or Price have negative values - those are likely returns/cancellations
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


### Check Missing Values

In [7]:
# Counts missing (empty) values in each column
df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
Price,0
Customer ID,243007
Country,0


### Check Duplicate Rows

In [8]:
# Counts how many rows are exact duplicates
duplicate_count = df.duplicated().sum()
print("Duplicate rows count:", duplicate_count)

Duplicate rows count: 34335


# Phase 02 - Data Cleaning

### Remove Duplicate Rows

In [9]:
# Remove exact duplicate rows, keep the first occurrence
print("Before removing duplicates:", df.shape)

df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

Before removing duplicates: (1067371, 8)
After removing duplicates: (1033036, 8)


### Handle Missing Customer ID

In [10]:
# Customer ID is essential for customer-level analysis (like RFM segmentation later)
# Rows without a Customer ID can't be linked to a specific customer, so we remove them
print("Before removing missing Customer ID:", df.shape)

df = df.dropna(subset=["Customer ID"])

print("After removing missing Customer ID:", df.shape)

Before removing missing Customer ID: (1033036, 8)
After removing missing Customer ID: (797885, 8)


### Handle Missing Description

In [11]:
# A few rows have missing product descriptions
# Since it's a small number, we can safely drop these rows too
print("Before removing missing Description:", df.shape)

df = df.dropna(subset=["Description"])

print("After removing missing Description:", df.shape)

Before removing missing Description: (797885, 8)
After removing missing Description: (797885, 8)


### Separate Cancelled Orders (Negative Quantity)

In [12]:
# Invoices starting with "C" are cancellations/returns (this is how the dataset marks them)
# We save these separately in case we want to analyze returns later,
# then keep only valid sales transactions in the main dataframe

df["Invoice"] = df["Invoice"].astype(str)

cancelled_orders = df[df["Invoice"].str.startswith("C")]
print("Cancelled/returned orders:", cancelled_orders.shape[0])

df = df[~df["Invoice"].str.startswith("C")]
print("Remaining valid sales orders:", df.shape[0])

Cancelled/returned orders: 18390
Remaining valid sales orders: 779495


### Remove Invalid Prices

In [13]:
# Price should never be 0 or negative for a real sale
print("Before removing invalid prices:", df.shape)

df = df[df["Price"] > 0]

print("After removing invalid prices:", df.shape)

Before removing invalid prices: (779495, 8)
After removing invalid prices: (779425, 8)


### Fix Data Types

In [14]:
# Convert InvoiceDate from text to proper datetime format
# This lets us do date-based analysis later (monthly trends, etc.)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Confirm the change
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 779425 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      779425 non-null  object        
 1   StockCode    779425 non-null  object        
 2   Description  779425 non-null  object        
 3   Quantity     779425 non-null  int64         
 4   InvoiceDate  779425 non-null  datetime64[ns]
 5   Price        779425 non-null  float64       
 6   Customer ID  779425 non-null  float64       
 7   Country      779425 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 53.5+ MB


### Create a Useful New Column

In [15]:
# TotalPrice = how much revenue this line item generated
# This will be very useful for sales analysis in the next phase
df["TotalPrice"] = df["Quantity"] * df["Price"]

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


### Final Check & Save Cleaned Data

In [16]:
# Confirm everything is clean now
print("Final shape:", df.shape)
print("\nRemaining missing values:\n", df.isnull().sum())
print("\nRemaining duplicates:", df.duplicated().sum())
print("\nMinimum Quantity:", df["Quantity"].min())
print("Minimum Price:", df["Price"].min())

Final shape: (779425, 9)

Remaining missing values:
 Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
TotalPrice     0
dtype: int64

Remaining duplicates: 0

Minimum Quantity: 1
Minimum Price: 0.001


In [17]:
# Save the cleaned dataset back to Google Drive so we can use it in Phase 3 (SQL)
output_path = "/content/drive/MyDrive/Retail Sales & Customer Insights Project/Dataset/cleaned_retail_data.csv"
df.to_csv(output_path, index=False)

print("Cleaned data saved to:", output_path)

Cleaned data saved to: /content/drive/MyDrive/Retail Sales & Customer Insights Project/Dataset/cleaned_retail_data.csv
